## data cleaning

notebook used to generate `.txt` files in `data/`

In [ ]:
# filter out puzzles with walls
with open("../data/rush.txt", "r", encoding="utf-8") as fin, open("../data/rush_no_walls.txt", "w", encoding="utf-8") as fout:
    for line in fin:
        stripped = line.strip()
        if not stripped:
            continue

        parts = stripped.split(maxsplit=2)
        if len(parts) != 3:
            continue

        _, board_desc, _ = parts
        if "x" not in board_desc:
            fout.write(line)

In [ ]:
# get 4x4 puzzles
import pandas as pd

df = pd.read_json("hf://datasets/mustafaah/rushhour4x4-eval/dataset.json")

In [ ]:
import re
import json

pattern = re.compile(r'Current Grid State \(JSON format\):\s*(\[\[.*?\]\])', re.DOTALL)

puzzles_4 = []
for prompt in df["prompt"].dropna():
    m = pattern.search(prompt)
    if not m:
        continue
    try:
        grid = json.loads(m.group(1))
        puzzles_4.append(grid)
    except json.JSONDecodeError:
        continue

In [ ]:
# format and load 4x4 puzzles into their own file
def grid_to_string(grid):
    result = []
    for row in grid:
        for cell in row:
            if isinstance(cell, str) and cell.startswith('H'):
                return None
            if cell == '.':
                result.append('o')
            elif cell == 'C':
                result.append('A')
            elif cell.startswith('B'):
                num = int(cell[1:])
                result.append(chr(ord('B') + num - 1))
            elif cell.startswith('H'):
                num = int(cell[1:])
                result.append(chr(ord('B') + num - 1))
            else:
                result.append(cell)
    return ''.join(result)

with open("../data/rush_4.txt", "w", encoding="utf-8") as f:
    for puzzle in puzzles_4:
        write_string = grid_to_string(puzzle)
        if write_string:
            f.write(write_string+ "\n")